LDA Model with Z-scoring

In [30]:
# MICrONS Visual Decoding with Z-scoring

!pip install -q dandi remfile pynwb h5py scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from scipy import stats

# ============================================================================
# CONNECT TO DATA
# ============================================================================

def connect_to_data():
    """Connect to the dataset and return the nwb file."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset("000402", "draft")
    all_assets = dandiset.get_assets()

    # Find the nwb file
    asset = None
    for a in all_assets:
        if a.path.endswith(".nwb"):
            asset = a
            break

    # Get the file URL and open it
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()
    return nwb


def get_neural_data(nwb):
    """Get the neural recording data."""
    ophys = nwb.processing["ophys"]
    fluorescence = ophys.data_interfaces["Fluorescence"]
    rs = fluorescence.roi_response_series["RoiResponseSeries3"]

    print("Total neurons:", rs.data.shape[1])
    print("Total timepoints:", rs.data.shape[0])

    return rs


def get_stimulus_info(nwb):
    """Get information about when stimuli were shown."""
    clip_intervals = nwb.intervals['Clip']

    clip_starts = np.array(clip_intervals.start_time[:])
    clip_stops = np.array(clip_intervals.stop_time[:])
    clip_types = np.array(clip_intervals.short_movie_name[:])

    print("Total stimulus presentations:", len(clip_starts))

    # Count how many of each type
    unique_types = np.unique(clip_types)
    for stim_type in unique_types:
        count = np.sum(clip_types == stim_type)
        print(f"  {stim_type}: {count} presentations")

    return clip_starts, clip_stops, clip_types


# ============================================================================
# EXTRACT AND PROCESS DATA
# ============================================================================

def extract_neural_features(rs, timestamps, clip_starts, clip_stops, clip_types,
                           neuron_list, target_conditions, window_duration=None):
    """
    Extract neural features for decoding with z-scoring.

    For each trial (clip presentation):
    1. Identify the time window (clip start to clip end or fixed duration)
    2. Convert times to indices in the neural data
    3. For each neuron, extract all timepoints in that window
    4. Calculate the mean across timepoints
    5. Apply z-scoring to normalize neural activity

    Parameters:
    - rs: neural recording data
    - timestamps: time values for each data point
    - clip_starts, clip_stops: when each stimulus started/stopped
    - clip_types: what type each stimulus was
    - neuron_list: list of neuron indices to use
    - target_conditions: which stimulus types to include
    - window_duration: if specified, use fixed duration (seconds) instead of full clip

    Returns:
    - X: z-scored neural data matrix (trials x neurons)
    - y: labels for each trial
    - condition_names: mapping from condition to label number
    """

    # Create condition to label mapping
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}

    all_trials = []
    all_labels = []

    # Process each stimulus presentation
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]

        # Skip if not in our target conditions
        if stim_type not in target_conditions:
            continue

        # 1. Identify the time window: when clip starts and when clip ends
        start_time = clip_starts[i]
        if window_duration is None:
            # Use full clip duration
            stop_time = clip_stops[i]
        else:
            # Use fixed duration from start
            stop_time = start_time + window_duration

        # 2. Convert times to indices - neural data is sampled at discrete timepoints
        # Find the first timepoint >= start_time
        start_idx = np.searchsorted(timestamps, start_time)

        # Find the first timepoint >= stop_time
        stop_idx = np.searchsorted(timestamps, stop_time)

        # 3. Extract neural data for this time window
        # Shape: (timepoints_in_window, num_neurons)
        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])

        # 4. For each neuron, take the mean across all timepoints
        # Get all timepoints for one neuron, calculate the mean
        neural_means = []
        for neuron_idx in range(neural_chunk.shape[1]):
            neuron_timepoints = neural_chunk[:, neuron_idx]
            mean_activity = np.mean(neuron_timepoints)
            neural_means.append(mean_activity)

        all_trials.append(neural_means)
        all_labels.append(condition_names[stim_type])

    # Convert to arrays
    X = np.array(all_trials)  # Shape: (trials, neurons)
    y = np.array(all_labels)

    # 5. Apply z-scoring: normalize each neuron's activity across trials
    # This helps with the mean by standardizing to zero mean, unit variance
    X_zscored = stats.zscore(X, axis=0)

    return X_zscored, y, condition_names


# ============================================================================
# TRAIN AND TEST DECODER
# ============================================================================

def train_and_evaluate_decoder(X, y, condition_names):
    """
    Train LDA decoder and evaluate performance.

    Parameters:
    - X: neural data (trials x neurons)
    - y: labels
    - condition_names: dict mapping condition names to numbers

    Returns:
    - accuracy: classification accuracy
    - confusion_info: detailed results
    """

    # Split into train (80%) and test (20%)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"\nTraining trials: {len(y_train)}")
    print(f"Test trials: {len(y_test)}")

    # Train Linear Discriminant Analysis decoder
    lda = LinearDiscriminantAnalysis()
    lda.fit(X_train, y_train)

    # Predict on test set
    y_pred = lda.predict(X_test)

    # Calculate accuracy
    accuracy = np.mean(y_test == y_pred)

    print(f"\nTest accuracy: {accuracy*100:.1f}%")

    # Per-condition accuracy
    print("\nPer-condition accuracy:")
    number_to_name = {v: k for k, v in condition_names.items()}

    for label_num in sorted(number_to_name.keys()):
        cond_name = number_to_name[label_num]

        # Find trials for this condition
        mask = (y_test == label_num)
        if np.sum(mask) > 0:
            cond_accuracy = np.mean(y_pred[mask] == y_test[mask])
            n_correct = np.sum(y_pred[mask] == y_test[mask])
            n_total = np.sum(mask)
            print(f"  {cond_name}: {cond_accuracy*100:.1f}% ({n_correct}/{n_total})")

    return accuracy, y_test, y_pred


# ============================================================================
# MAIN ANALYSIS
# ============================================================================

# Load data
nwb = connect_to_data()
rs = get_neural_data(nwb)
clip_starts, clip_stops, clip_types = get_stimulus_info(nwb)

# Get timestamps (using first 100,000 samples for manageable size)
timestamps = np.array(rs.timestamps[:100000])

# Define which visual stimulus types to decode
target_conditions = ['Cinematic', 'Rendered', 'sports1m']

# Select approximately 200 neurons from the population
n_neurons = 200
neuron_list = list(range(n_neurons))

# ============================================================================
# TEST 1: Full clip duration
# ============================================================================

print(f"\nTest 1: Using Full Clip Duration")
print(f"Using {n_neurons} neurons to decode {len(target_conditions)} stimulus types")

# Extract and z-score neural features using full clip duration
X_full, y_full, condition_names = extract_neural_features(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list, target_conditions, window_duration=None
)

# Train decoder and evaluate
accuracy_full, y_test_full, y_pred_full = train_and_evaluate_decoder(
    X_full, y_full, condition_names
)

# ============================================================================
# TEST 2: First 2 seconds only
# ============================================================================
print(f"\nTest 2: Using first 2 seconds only")
print(f"Using {n_neurons} neurons to decode {len(target_conditions)} stimulus types")


# Extract and z-score neural features using only first 2 seconds
X_2sec, y_2sec, condition_names = extract_neural_features(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list, target_conditions, window_duration=2.0
)

# Train decoder and evaluate
accuracy_2sec, y_test_2sec, y_pred_2sec = train_and_evaluate_decoder(
    X_2sec, y_2sec, condition_names
)


Total neurons: 1455
Total timepoints: 40000
Total stimulus presentations: 384
  Cinematic: 128 presentations
  Rendered: 128 presentations
  sports1m: 128 presentations

Test 1: Using Full Clip Duration
Using 200 neurons to decode 3 stimulus types

Training trials: 307
Test trials: 77

Test accuracy: 57.1%

Per-condition accuracy:
  Cinematic: 52.0% (13/25)
  Rendered: 61.5% (16/26)
  sports1m: 57.7% (15/26)

Test 2: Using first 2 seconds only
Using 200 neurons to decode 3 stimulus types

Training trials: 307
Test trials: 77

Test accuracy: 36.4%

Per-condition accuracy:
  Cinematic: 44.0% (11/25)
  Rendered: 30.8% (8/26)
  sports1m: 34.6% (9/26)


Logistic Regression with Z scoring and K-fold


In [29]:
# MICrONS Visual Decoding with K-Fold Cross-Validation

!pip install -q dandi remfile pynwb h5py scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy import stats

# ============================================================================
# CONNECT TO DATA
# ============================================================================

def connect_to_data():
    """Connect to the dataset and return the nwb file."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset("000402", "draft")
    all_assets = dandiset.get_assets()

    asset = None
    for a in all_assets:
        if a.path.endswith(".nwb"):
            asset = a
            break

    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()

    return nwb


def get_neural_data(nwb):
    """Get the neural recording data."""
    ophys = nwb.processing["ophys"]
    fluorescence = ophys.data_interfaces["Fluorescence"]
    rs = fluorescence.roi_response_series["RoiResponseSeries3"]

    print("Total neurons:", rs.data.shape[1])
    print("Total timepoints:", rs.data.shape[0])

    return rs


def get_stimulus_info(nwb):
    """Get information about when stimuli were shown."""
    clip_intervals = nwb.intervals['Clip']

    clip_starts = np.array(clip_intervals.start_time[:])
    clip_stops = np.array(clip_intervals.stop_time[:])
    clip_types = np.array(clip_intervals.short_movie_name[:])

    print("Total stimulus presentations:", len(clip_starts))

    unique_types = np.unique(clip_types)
    for stim_type in unique_types:
        count = np.sum(clip_types == stim_type)
        print(f"  {stim_type}: {count} presentations")

    return clip_starts, clip_stops, clip_types


# ============================================================================
# EXTRACT AND PROCESS DATA
# ============================================================================

def extract_neural_features(rs, timestamps, clip_starts, clip_stops, clip_types,
                           neuron_list, target_conditions, window_duration=None):
    """Extract neural features for decoding with z-scoring."""

    condition_names = {cond: i for i, cond in enumerate(target_conditions)}

    all_trials = []
    all_labels = []

    for i in range(len(clip_starts)):
        stim_type = clip_types[i]

        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        if window_duration is None:
            stop_time = clip_stops[i]
        else:
            stop_time = start_time + window_duration

        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])
        neural_means = np.mean(neural_chunk, axis=0)

        all_trials.append(neural_means)
        all_labels.append(condition_names[stim_type])

    X = np.array(all_trials)
    y = np.array(all_labels)
    X_zscored = stats.zscore(X, axis=0)

    return X_zscored, y, condition_names


# ============================================================================
# TRAIN AND TEST DECODER
# ============================================================================

def train_and_evaluate_decoder_kfold(X, y, condition_names, n_folds=5,
                                    C=1.0, penalty='l2', solver='lbfgs',
                                    class_weight=None, max_iter=1000):
    """Train Logistic Regression decoder with k-fold cross-validation."""

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    fold_accuracies = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train = X[train_idx]
        X_test = X[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        logreg = LogisticRegression(
            C=C,
            penalty=penalty,
            solver=solver,
            class_weight=class_weight,
            max_iter=max_iter,
            random_state=42
        )
        logreg.fit(X_train, y_train)

        y_pred = logreg.predict(X_test)
        fold_accuracy = np.mean(y_test == y_pred)
        fold_accuracies.append(fold_accuracy)

        print(f"Fold {fold_idx + 1}: {fold_accuracy*100:.1f}%")

    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)

    print(f"Mean: {mean_accuracy*100:.1f}% ± {std_accuracy*100:.1f}%")

    return mean_accuracy, fold_accuracies


# ============================================================================
# MAIN ANALYSIS
# ============================================================================

print("Neural Decoding Analysis\n")

# Load data
nwb = connect_to_data()
rs = get_neural_data(nwb)
clip_starts, clip_stops, clip_types = get_stimulus_info(nwb)

timestamps = np.array(rs.timestamps[:100000])
target_conditions = ['Cinematic', 'Rendered', 'sports1m']

n_neurons = 200
neuron_list = list(range(n_neurons))
n_folds = 5

# Extract data
X_full, y_full, condition_names = extract_neural_features(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list, target_conditions, window_duration=None
)

X_2sec, y_2sec, _ = extract_neural_features(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list, target_conditions, window_duration=2.0
)

# ============================================================================
# TESTS
# ============================================================================

print("\nTest 1: Default (C=1.0, L2)")
print("Full clip:")
accuracy_1_full, _ = train_and_evaluate_decoder_kfold(
    X_full, y_full, condition_names, n_folds=n_folds,
    C=1.0, penalty='l2', solver='lbfgs', max_iter=1000
)

print("\nFirst 2 seconds:")
accuracy_1_2sec, _ = train_and_evaluate_decoder_kfold(
    X_2sec, y_2sec, condition_names, n_folds=n_folds,
    C=1.0, penalty='l2', solver='lbfgs', max_iter=1000
)

print("\n" + "="*50)
print("Test 2: Strong regularization (C=0.01, L2)")
print("Full clip:")
accuracy_2_full, _ = train_and_evaluate_decoder_kfold(
    X_full, y_full, condition_names, n_folds=n_folds,
    C=0.01, penalty='l2', solver='lbfgs', max_iter=1000
)

print("\nFirst 2 seconds:")
accuracy_2_2sec, _ = train_and_evaluate_decoder_kfold(
    X_2sec, y_2sec, condition_names, n_folds=n_folds,
    C=0.01, penalty='l2', solver='lbfgs', max_iter=1000
)

print("\n" + "="*50)
print("Test 3: Weak regularization (C=10, L2)")
print("Full clip:")
accuracy_3_full, _ = train_and_evaluate_decoder_kfold(
    X_full, y_full, condition_names, n_folds=n_folds,
    C=10, penalty='l2', solver='lbfgs', max_iter=1000
)

print("\nFirst 2 seconds:")
accuracy_3_2sec, _ = train_and_evaluate_decoder_kfold(
    X_2sec, y_2sec, condition_names, n_folds=n_folds,
    C=10, penalty='l2', solver='lbfgs', max_iter=1000
)

print("\n" + "="*50)
print("Test 4: L1 regularization")
print("Full clip:")
accuracy_4_full, _ = train_and_evaluate_decoder_kfold(
    X_full, y_full, condition_names, n_folds=n_folds,
    C=1.0, penalty='l1', solver='liblinear', max_iter=1000
)

print("\nFirst 2 seconds:")
accuracy_4_2sec, _ = train_and_evaluate_decoder_kfold(
    X_2sec, y_2sec, condition_names, n_folds=n_folds,
    C=1.0, penalty='l1', solver='liblinear', max_iter=1000
)

print("\n" + "="*50)
print("Test 5: No regularization")
print("Full clip:")
accuracy_5_full, _ = train_and_evaluate_decoder_kfold(
    X_full, y_full, condition_names, n_folds=n_folds,
    C=1.0, penalty=None, solver='lbfgs', max_iter=1000
)

print("\nFirst 2 seconds:")
accuracy_5_2sec, _ = train_and_evaluate_decoder_kfold(
    X_2sec, y_2sec, condition_names, n_folds=n_folds,
    C=1.0, penalty=None, solver='lbfgs', max_iter=1000
)

Neural Decoding Analysis

Total neurons: 1455
Total timepoints: 40000
Total stimulus presentations: 384
  Cinematic: 128 presentations
  Rendered: 128 presentations
  sports1m: 128 presentations

Test 1: Default (C=1.0, L2)
Full clip:
Fold 1: 55.8%
Fold 2: 63.6%
Fold 3: 66.2%
Fold 4: 67.5%
Fold 5: 59.2%
Mean: 62.5% ± 4.4%

First 2 seconds:
Fold 1: 44.2%
Fold 2: 45.5%
Fold 3: 48.1%
Fold 4: 46.8%
Fold 5: 38.2%
Mean: 44.5% ± 3.4%

Test 2: Strong regularization (C=0.01, L2)
Full clip:
Fold 1: 58.4%
Fold 2: 62.3%
Fold 3: 68.8%
Fold 4: 72.7%
Fold 5: 53.9%
Mean: 63.3% ± 6.8%

First 2 seconds:
Fold 1: 46.8%
Fold 2: 49.4%
Fold 3: 51.9%
Fold 4: 51.9%
Fold 5: 43.4%
Mean: 48.7% ± 3.3%

Test 3: Weak regularization (C=10, L2)
Full clip:
Fold 1: 51.9%
Fold 2: 61.0%
Fold 3: 64.9%
Fold 4: 67.5%
Fold 5: 59.2%
Mean: 60.9% ± 5.4%

First 2 seconds:
Fold 1: 45.5%
Fold 2: 46.8%
Fold 3: 44.2%
Fold 4: 45.5%
Fold 5: 35.5%
Mean: 43.5% ± 4.1%

Test 4: L1 regularization
Full clip:
Fold 1: 58.4%
Fold 2: 64.9%
Fold 